# zero-grad-set-none — ex2: diagnose a training loop where zero_grad runs BEFORE step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `zero-grad-set-none`. Running the final beacon cell reports progress against the `PyTorch: zero_grad` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: zero_grad` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`zero-grad-set-none`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "zero-grad-set-none"
DD_SUBTOPIC = "PyTorch: zero_grad"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## zero_grad — when does it go in the loop?

The canonical training step has FOUR statements in a fixed order:

```python
for xb, yb in loader:
    loss = loss_fn(model(xb), yb)
    loss.backward()       # 1. compute gradients
    optimizer.step()      # 2. apply update
    optimizer.zero_grad() # 3. wipe grads for the NEXT step
```

Equivalently, `zero_grad()` can be the FIRST statement of the NEXT iteration — but it MUST sit between `step` of iteration N and `backward` of iteration N+1. Putting it BETWEEN `backward` and `step` of the same iteration is the **classic silent bug**: gradients are wiped before the optimizer can read them, so `.step()` becomes a no-op and the model never learns.

The previous drill (ex1) implemented the *body* of `zero_grad` (`p.grad = None` for each param). This drill targets the **ORDERING** — given a buggy training loop, identify which placement is correct and fix it.

### Exercise 2 — diagnose a training loop where zero_grad runs BEFORE step

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the relative ordering of `backward`, `step`, and `zero_grad` in a training loop and fix a variant in which `zero_grad` was placed BEFORE `step` so the optimizer would have nothing to read.
> Keywords: training-loop, zero-grad, ordering, silent-bug, diagnose
> ```

**KCs targeted:** `zero-grad-set-to-none-semantics`, `zero-grad-must-follow-step`

Below is a buggy training step. The author put `optimizer.zero_grad()` between `backward()` and `step()` — so the gradients computed by `backward` are wiped to None BEFORE `step` runs, and the parameters never get updated. The loss never decreases.

Implement `ex2_fixed_step(model, opt, x, y, loss_fn)` — ONE call to the fixed training step. The fix is to set `p.grad = None` AFTER `step()`, not between `backward()` and `step()`. The function returns the post-step loss value (the float scalar from before the update — i.e. the value of `loss` that the gradients were computed against).

**Required order:**
1. `pred = model(x)`
2. `loss = loss_fn(pred, y)`
3. `loss.backward()`
4. `opt.step()`
5. for `p in model.parameters(): p.grad = None`  ← AFTER step
6. return `loss.item()`

**Why the bug is silent.** No exception, no warning. The model's loss just stays flat across epochs. Detection requires logging the loss trajectory — if it's literally constant when it should be falling, this is the first place to look.

The test runs the fixed step many times on a quadratic loss and confirms the loss decreases monotonically — then runs the buggy version (provided below) and confirms the loss is constant.

In [ ]:
def ex2_buggy_step(model, opt, x, y, loss_fn):
    # The bug — kept as a reference foil.
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    for p in model.parameters():
        p.grad = None
    opt.step()
    return loss.item()


def ex2_fixed_step(model, opt, x, y, loss_fn) -> float:
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    for p in model.parameters():
        p.grad = None
    return loss.item()


<details><summary>Solution</summary>

```python
def ex2_buggy_step(model, opt, x, y, loss_fn):
    # The bug — kept as a reference foil.
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    for p in model.parameters():
        p.grad = None
    opt.step()
    return loss.item()


def ex2_fixed_step(model, opt, x, y, loss_fn) -> float:
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    for p in model.parameters():
        p.grad = None
    return loss.item()
```

**The standard tripwire.** Every PyTorch tutorial places `zero_grad` at one of two correct spots: (a) the LAST line of the training step (after `step`), or (b) the FIRST line of the next iteration (before `backward`). Anywhere ELSE is wrong. The 'between backward and step' placement looks innocent — same set of three function names, just reordered — but it silently neuters the optimizer.

**Why it's silent.** `step` reads `param.grad`; if it's `None`, it does nothing for that param (no exception). With `set_to_none=True` this is the explicit contract — `None` means 'no gradient yet, skip.'

**Difference from ex1.** ex1 implemented the BODY of `zero_grad` (set each `.grad` to `None`). ex2 targets the ORDERING — diagnosing and fixing a training loop where `zero_grad` ran at the wrong moment.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()